# EXPERIMENT 5A – Random Forest Classifier for Car Safety

**Aim:** Predict the safety level of a car using a Random Forest Classifier.

**Dataset:** [Kaggle Car Evaluation](https://www.kaggle.com/datasets/elikplim/car-evaluation-data-set)

**Target:** `safety` (low, med, high). **Input features:** buying, maint, doors, persons and lug_boot.

**Dataset clarification:** The CSV also contains `class`, which represents overall car acceptability (unacc, acc, good, vgood). Since this experiment asks to predict *safety*, this notebook uses `safety` as the target and excludes `class` from the inputs.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

## 2. Load the dataset

Download the Kaggle CSV and save it in the same folder as this notebook as `car_evaluation.csv`. The code also supports the original headerless `car.data` filename.

In [ ]:
columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]

try:
    cars = pd.read_csv("car_evaluation.csv")
    if not set(columns).issubset(cars.columns):
        cars = pd.read_csv("car_evaluation.csv", header=None, names=columns)
except FileNotFoundError:
    cars = pd.read_csv("car.data", header=None, names=columns)

# Remove extra spaces from column names and categorical values
cars.columns = cars.columns.astype(str).str.strip()
for col in cars.select_dtypes(include="object").columns:
    cars[col] = cars[col].str.strip()

print("Dataset shape:", cars.shape)
display(cars.head())

## 3. Explore the dataset

In [ ]:
print("Columns:", cars.columns.tolist())
print("\nMissing values:\n", cars.isnull().sum())
print("\nSafety level counts:\n", cars["safety"].value_counts())

cars["safety"].value_counts().plot(kind="bar", rot=0)
plt.title("Distribution of Car Safety Levels")
plt.xlabel("Safety")
plt.ylabel("Number of records")
plt.show()

## 4. Select features and target
We use five car attributes to predict the safety category. The `class` column is excluded to avoid target leakage.

In [ ]:
X = cars[["buying", "maint", "doors", "persons", "lug_boot"]]
y = cars["safety"]

print("Input features:", X.columns.tolist())
print("Target classes:", sorted(y.unique()))

## 5. Split into training and testing data
80% is used for training and 20% for testing. Stratification preserves the safety-class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 6. Encode categorical features and create Random Forest
OneHotEncoder converts text categories into numeric columns. The Pipeline ensures the encoder is fitted only on training data.

In [ ]:
# Encode the categorical input columns
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), X.columns)
])

# Random Forest combines the predictions of many decision trees
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf)
])

## 7. Train the model

In [ ]:
model.fit(X_train, y_train)
print("Random Forest training completed.")

## 8. Predict safety levels

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual safety": y_test.to_numpy()[:10],
    "Predicted safety": y_pred[:10]
})
display(comparison)

## 9. Evaluate the classifier
Accuracy is the proportion of correct predictions. The classification report provides precision, recall and F1-score for each safety category.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")

labels = ["low", "med", "high"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("\nConfusion matrix (actual rows, predicted columns):\n", cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))

## 10. Confusion matrix visualization

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", values_format="d")
plt.title("Random Forest – Car Safety Prediction")
plt.show()

## 11. Predict for a new car

In [ ]:
# Example car details; change the values to test another car
new_car = pd.DataFrame([{
    "buying": "med",
    "maint": "low",
    "doors": "4",
    "persons": "4",
    "lug_boot": "big"
}])

print("Predicted safety:", model.predict(new_car)[0])

## 12. Conclusion

Random Forest was implemented to predict the car's safety level using five categorical car attributes. One-hot encoding was used to convert text categories into numeric features. The model was evaluated using accuracy, confusion matrix, precision, recall and F1-score.

**Result:** The predictions and evaluation metrics are displayed after running the notebook.